In [25]:
import pandas as pd
import altair as alt

df = pd.read_excel('data/SCM_Dataset_Updated_with_Green_Logistics.xlsx')
df = df.dropna(subset=['Operational Efficiency Score', 'SCM Practices'])

# unique practices
unique_practices = sorted(df['SCM Practices'].unique().tolist())

checkboxes = []
practice_params = {}

# making an "All" checkbox
all_param = alt.param(name="all_selected", value=True, bind=alt.binding_checkbox(name=" All SCM Practices"))
checkboxes.append(all_param)

# checkboxes for each practice
for practice in unique_practices:
    safe_name = f"practice_{practice.replace(' ', '_').replace('-', '_')}"
    param = alt.param(name=safe_name, value=False, bind=alt.binding_checkbox(name=f" {practice} "))
    practice_params[practice] = param
    checkboxes.append(param)

chart = alt.Chart(df).mark_boxplot(size=60, color='steelblue').encode(
    x=alt.X('SCM Practices:N', axis=alt.Axis(labelAngle=-30)),
    y=alt.Y('Operational Efficiency Score:Q', title='Operational Efficiency', scale=alt.Scale(domain=[70, 95]))
).properties(
    width=785,
    height=450,
    title='Operational Efficiency by SCM Practice'
)

# filter condition that shows all practices when "All" is selected and only specifically selected practices otherwise
filter_conditions = [f"{all_param.name}"]
for practice, param in practice_params.items():
    filter_conditions.append(f"(datum['SCM Practices'] === '{practice}' && {param.name})")

filter_expression = " || ".join(filter_conditions)

final_chart = chart.add_params(
    *checkboxes
).transform_filter(
    filter_expression
)
final_chart.save('scm_boxplot.html')